**eBPF vs strace dependency tracing.** The eBPF backend captures workspace read and negative-lookup effects from Linux syscall tracepoints instead of ptrace. This figure compares per-step cost (mean/p50/p95), the incremental tracing tax over no tracing, and dependency-capture fidelity. Data comes from `experiments/scripts/bench_bpf_trace.py` (root + bpftrace required).

In [ ]:
# ipython -c "%run plot_bpf_trace.ipynb"

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

STANDARD_WIDTH = 17.8
MODE_ORDER = ['off', 'strace', 'bpf']
METRIC_STYLES = {
    'per_step_ms_mean': dict(color='#c00000', marker='s', linestyle='-', linewidth=1.0, markersize=3.2),
    'per_step_ms_p50': dict(color='#7f7f7f', marker='o', linestyle='--', linewidth=0.8, markersize=2.8, markerfacecolor='none'),
    'per_step_ms_p95': dict(color='#e78129', marker='x', linestyle=':', linewidth=0.9, markersize=3.6, markeredgewidth=0.9),
}
LABELS = {
    'per_step_ms_mean': 'mean',
    'per_step_ms_p50': 'p50',
    'per_step_ms_p95': 'p95',
}
MODE_LABELS = {
    'off': 'no tracing',
    'strace': 'strace',
    'bpf': 'eBPF',
}

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
result = RESULTS / 'bpf_trace_overhead.csv'
if not result.exists():
    raise FileNotFoundError(
        f'missing {result}; run experiments/scripts/bench_bpf_trace.py '
        'as root on a host with bpftrace first'
    )
df = pd.read_csv(result)
numeric = [
    'per_step_ms_mean', 'per_step_ms_p50', 'per_step_ms_p95',
    'reads_captured', 'negatives_captured', 'capture_expected',
]
df[numeric] = df[numeric].apply(pd.to_numeric)

baseline = df[df['mode'] == 'off']['per_step_ms_mean'].iloc[0]
df = df.sort_values('mode', key=lambda s: s.map(MODE_ORDER.index))
x = [float(MODE_ORDER.index(mode)) for mode in df['mode']]

def values(metric):
    return df[metric].tolist()

fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(7.0)))
handles = []

ax = plt.subplot(2, 2, 1)
for metric in ('per_step_ms_mean', 'per_step_ms_p50', 'per_step_ms_p95'):
    handle, = ax.plot(x, values(metric), **METRIC_STYLES[metric], label=LABELS[metric])
    handles.append(handle)
ax.set_ylabel('Per-step cost (ms)', fontsize=8)
ax.set_xlabel('Tracing backend\n(a) Per-step cost', fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels([MODE_LABELS[m] for m in df['mode']], fontsize=6.5)
ax.tick_params(axis='both', labelsize=7)

incremental = {
    mode: (df[df['mode'] == mode]['per_step_ms_mean'].iloc[0] - baseline)
    for mode in ('strace', 'bpf')
}
ax = plt.subplot(2, 2, 2)
ix = [float(MODE_ORDER.index(m)) for m in incremental]
handle, = ax.plot(ix, list(incremental.values()), **METRIC_STYLES['per_step_ms_mean'],
                  label='incremental cost')
handles.append(handle)
ax.axhline(0.0, color='0.6', linewidth=0.6, linestyle=':')
ax.set_ylabel('Incremental cost (ms)', fontsize=8)
ax.set_xlabel('Tracing backend\n(b) Tracing tax vs no tracing', fontsize=7)
ax.set_xticks(ix)
ax.set_xticklabels([MODE_LABELS[m] for m in incremental], fontsize=6.5)
ax.tick_params(axis='both', labelsize=7)

traced = df[df['mode'] != 'off']
ax = plt.subplot(2, 2, 3)
x_traced = [float(MODE_ORDER.index(m)) for m in traced['mode']]
expected = traced['capture_expected'].astype(float)
rate = 100.0 * (traced['reads_captured'] + traced['negatives_captured']) / (2.0 * expected)
handle, = ax.plot(x_traced, rate.tolist(), **METRIC_STYLES['per_step_ms_p95'],
                  label='read + negative capture')
handles.append(handle)
ax.axhline(100.0, color='0.6', linewidth=0.6, linestyle=':')
ax.set_ylim(0, 105)
ax.set_ylabel('Capture rate (%)', fontsize=8)
ax.set_xlabel('Tracing backend\n(c) Dependency capture fidelity', fontsize=7)
ax.set_xticks(x_traced)
ax.set_xticklabels([MODE_LABELS[m] for m in traced['mode']], fontsize=6.5)
ax.tick_params(axis='both', labelsize=7)

ax = plt.subplot(2, 2, 4)
strace_p95 = df[df['mode'] == 'strace']['per_step_ms_p95'].iloc[0]
bpf_p95 = df[df['mode'] == 'bpf']['per_step_ms_p95'].iloc[0]
ratio = (strace_p95 - bpf_p95) / max(strace_p95, 1e-9) * 100.0
ax.text(0.05, 0.55, f'p95 ratio (strace − eBPF):\n{ratio:+.1f}% of strace p95',
        transform=ax.transAxes, fontsize=8, va='center')
ax.text(0.05, 0.30,
        'Verified: every traced step captured\nboth the READ and the NEGATIVE\neffect in the ledger.',
        transform=ax.transAxes, fontsize=7, color='0.25', va='center')
ax.set_xlabel('(d) Interpretation', fontsize=7)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.035), ncol=4,
           fontsize=7, columnspacing=1.0, handlelength=1.8, handletextpad=0.35, borderpad=0.3)
plt.tight_layout(pad=0.6, h_pad=1.6, w_pad=1.2, rect=[0.0, 0.0, 1.0, 0.91])
plt.savefig(FIGDIR / 'FIG-Bpf-Trace.pdf', bbox_inches='tight', pad_inches=0.02,
            metadata={'CreationDate': None, 'ModDate': None})
plt.savefig(FIGDIR / 'FIG-Bpf-Trace.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

assert (df[df['mode'] != 'off']['reads_captured'] == df[df['mode'] != 'off']['capture_expected']).all()
assert (df[df['mode'] != 'off']['negatives_captured'] == df[df['mode'] != 'off']['capture_expected']).all()
